# Complete LAPD Crime Data Cleaning Workflow

This notebook combines the most useful ideas from the six team files into one reproducible workflow for:

1. `LAPD_NIBRS_Offenses_Dataset.csv` (current NIBRS offense-level data)
2. `Crime_Data_from_2020_to_2024.csv` (legacy crime-report data)

The datasets stay separate because their row definitions and schemas differ. The workflow preserves every source row, cleans values only when the rule is defensible, and adds transparent flags for questionable records instead of silently deleting or inventing data.

**How to run:** Put both CSV files in the same folder as this notebook (or in a `data/` subfolder), select a Python kernel with pandas installed, then choose **Restart Kernel and Run All Cells**. Change the paths in the configuration cell if your filenames differ.


## What was combined from the team files

- **Kirsten:** two-dataset workflow, column harmonization, yes/no standardization, coordinate/count/area/district checks, and 50-row previews.
- **Iyanna:** source preservation, explicit date-conversion checks, difficult-case sampling, reconciliation, and a cleaning decision log.
- **Vicky:** broad NIBRS audit, occurrence-hour derivation, status conversion, and preservation of repeated case numbers.
- **Project notebook:** reproducible full-data run and concise validation summary.
- **Steven and Yunbo notebooks:** assertions, external validation evidence, expected-vs-actual record checks, and clear distinction between cleaning and analytical eligibility.

Some submitted notebooks depend on external files that were not attached. This notebook is self-contained except for the two source CSV files.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "cleaned_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_NAMES = {
    "nibrs": "LAPD_NIBRS_Offenses_Dataset.csv",
    "legacy": "Crime_Data_from_2020_to_2024.csv",
}

def locate_file(filename):
    candidates = [PROJECT_ROOT / filename, PROJECT_ROOT / "data" / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Put it beside the notebook, put it in data/, "
        "or edit FILE_NAMES in this cell."
    )

SOURCE_PATHS = {name: locate_file(filename) for name, filename in FILE_NAMES.items()}
SOURCE_PATHS


FileNotFoundError: Could not find LAPD_NIBRS_Offenses_Dataset.csv. Put it beside the notebook, put it in data/, or edit FILE_NAMES in this cell.

## 1. Load source snapshots without changing them

Identifiers and categorical codes are loaded as strings when present so leading zeros are not lost. `raw` is never edited; all transformations occur on independent copies.


In [ ]:
STRING_COLUMNS = {
    "nibrs": ["UniqueNIBRNo", "CaseNo", "Status", "RPT Dist No"],
    "legacy": ["DR_NO", "Status", "Rpt Dist No", "Crm Cd 1", "Crm Cd 2", "Crm Cd 3", "Crm Cd 4"],
}

def load_source(path, requested_string_columns):
    header = pd.read_csv(path, nrows=0).columns
    dtype = {column: "string" for column in requested_string_columns if column in header}
    return pd.read_csv(path, dtype=dtype, low_memory=False)

raw = {
    name: load_source(SOURCE_PATHS[name], STRING_COLUMNS[name])
    for name in SOURCE_PATHS
}
clean = {name: frame.copy(deep=True) for name, frame in raw.items()}

source_inventory = pd.DataFrame([
    {
        "dataset": name,
        "source_file": str(SOURCE_PATHS[name]),
        "rows": len(frame),
        "columns": frame.shape[1],
        "memory_mb": round(frame.memory_usage(deep=True).sum() / 1_048_576, 1),
    }
    for name, frame in raw.items()
])
display(source_inventory)


## 2. Audit row grain, identifiers, duplicates, missing values, and types

Repeated `CaseNo` values are not automatically duplicates because a case may contain several distinct offenses. Exact duplicate rows and repeated candidate keys are reported separately.


In [ ]:
KEYS = {"nibrs": "UniqueNIBRNo", "legacy": "DR_NO"}

def audit_frame(frame, key):
    key_values = frame[key].dropna() if key in frame else pd.Series(dtype="string")
    return {
        "rows": len(frame),
        "columns": frame.shape[1],
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "missing_key_values": int(frame[key].isna().sum()) if key in frame else None,
        "distinct_key_values": int(key_values.nunique()),
        "rows_with_repeated_key": int(key_values.duplicated(keep=False).sum()),
    }

audit_before = pd.DataFrame([
    {"dataset": name, **audit_frame(frame, KEYS[name])}
    for name, frame in raw.items()
])
display(audit_before)

missing_before = {
    name: pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_percent": (frame.isna().mean() * 100).round(2),
    }).query("missing_count > 0").sort_values("missing_count", ascending=False)
    for name, frame in raw.items()
}
for name, report in missing_before.items():
    print(f"\n{name.upper()} columns containing missing values")
    display(report)


## 3. Standardize column names needed by both datasets

The original columns are renamed only where the two files use different capitalization or labels. This creates consistent names for shared validation functions while leaving dataset-specific fields intact.


In [ ]:
COMMON_RENAMES = {
    "Date Rptd": "Date_Reported",
    "DATE OCC": "Date_Occurred",
    "Date OCC": "Date_Occurred",
    "TIME OCC": "Time_Occurred",
    "Time OCC": "Time_Occurred",
    "AREA": "Area_No",
    "AREA NAME": "Area_Name",
    "Rpt Dist No": "Report_District_No",
    "RPT Dist No": "Report_District_No",
    "Crm Cd": "Crime_Code",
    "Crm Cd Desc": "Crime_Description",
    "Mocodes": "MO_Codes",
    "Vict Age": "Victim_Age",
    "Vict Sex": "Victim_Sex",
    "Vict Descent": "Victim_Descent",
    "Premis Cd": "Premise_Code",
    "Premis Desc": "Premise_Description",
    "LOCATION": "Location",
    "Cross Street": "Cross_Street",
}

for name in clean:
    clean[name].columns = clean[name].columns.str.strip()
    clean[name] = clean[name].rename(columns=COMMON_RENAMES)

for name, frame in clean.items():
    duplicated_names = frame.columns[frame.columns.duplicated()].tolist()
    assert not duplicated_names, f"{name}: duplicate columns after renaming: {duplicated_names}"
    print(name, frame.shape, "columns standardized")


## 4. Clean text and categorical values

All text receives outer-whitespace trimming and internal whitespace is collapsed. Empty strings become true missing values. Identifier case and NIBRS yes/no flags are standardized using explicit mappings; unexpected flag values remain missing and are counted for review.


In [ ]:
def clean_text_columns(frame):
    text_columns = frame.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        values = frame[column].astype("string")
        values = values.str.strip().str.replace(r"\s+", " ", regex=True)
        frame[column] = values.mask(values.eq(""), pd.NA)
    return frame

for name in clean:
    clean[name] = clean_text_columns(clean[name])

for column in ["UniqueNIBRNo", "CaseNo"]:
    if column in clean["nibrs"]:
        clean["nibrs"][column] = clean["nibrs"][column].str.upper()

YES_NO_COLUMNS = [
    "Victim Shot", "Domestic Violence Crime", "Hate Crime", "Gang-related Crime",
    "Transit-related Crime", "Homeless-Victim Crime", "Homeless-Suspect Crime",
    "Homeless-Arrestee Crime",
]
YES_NO_MAP = {
    "yes": "Yes", "y": "Yes", "true": "Yes", "1": "Yes",
    "no": "No", "n": "No", "false": "No", "0": "No",
}
flag_audit = []
for column in YES_NO_COLUMNS:
    if column not in clean["nibrs"]:
        continue
    original = clean["nibrs"][column]
    normalized = original.str.lower().map(YES_NO_MAP).astype("string")
    unexpected = original.notna() & normalized.isna()
    flag_audit.append({"column": column, "unexpected_values": int(unexpected.sum())})
    clean["nibrs"][column] = normalized

display(pd.DataFrame(flag_audit))


## 5. Parse dates and validate HHMM occurrence times

Failed conversions are flagged rather than hidden. Dates remain full pandas datetimes. Occurrence time is preserved as a nullable integer and is also converted to a readable `HH:MM` value and an `occurrence_hour` field.


In [ ]:
DATE_FORMAT = "%Y %b %d %I:%M:%S %p"

def parse_dates_and_time(frame):
    checks = []
    for column in ["Date_Reported", "Date_Occurred"]:
        if column not in frame:
            continue
        original = frame[column]
        parsed = pd.to_datetime(original, format=DATE_FORMAT, errors="coerce")
        # A controlled fallback handles already-ISO-formatted inputs without changing valid exact parses.
        needs_fallback = original.notna() & parsed.isna()
        if needs_fallback.any():
            parsed.loc[needs_fallback] = pd.to_datetime(original.loc[needs_fallback], errors="coerce")
        frame[f"flag_{column.lower()}_parse_failure"] = original.notna() & parsed.isna()
        checks.append({
            "column": column,
            "nonmissing_source": int(original.notna().sum()),
            "parse_failures": int((original.notna() & parsed.isna()).sum()),
        })
        frame[column] = parsed

    if "Time_Occurred" in frame:
        time_value = pd.to_numeric(frame["Time_Occurred"], errors="coerce").astype("Int64")
        valid = (
            time_value.notna() & time_value.between(0, 2359)
            & (time_value // 100).between(0, 23)
            & (time_value % 100).between(0, 59)
        )
        frame["flag_invalid_occurrence_time"] = frame["Time_Occurred"].notna() & ~valid
        frame["Time_Occurred"] = time_value
        frame["occurrence_hour"] = (time_value // 100).where(valid).astype("Int64")
        hour = (time_value // 100).astype("string").str.zfill(2)
        minute = (time_value % 100).astype("string").str.zfill(2)
        frame["Occurrence_Time_HHMM"] = (hour + ":" + minute).where(valid, pd.NA)
    return frame, pd.DataFrame(checks)

date_checks = {}
for name in clean:
    clean[name], date_checks[name] = parse_dates_and_time(clean[name])
    print(f"\n{name.upper()} date checks")
    display(date_checks[name])


## 6. Convert numeric analysis fields safely

The original invalid text is not silently discarded: every conversion receives a companion failure flag. Codes used only as identifiers remain strings.


In [ ]:
NUMERIC_COLUMNS = {
    "nibrs": ["Area_No", "Report_District_No", "LAT", "LON", "TotalOffenseCount", "TotalVictimCount"],
    "legacy": ["Area_No", "Report_District_No", "LAT", "LON", "Victim_Age"],
}

numeric_checks = []
for name, columns in NUMERIC_COLUMNS.items():
    for column in columns:
        if column not in clean[name]:
            continue
        original = clean[name][column]
        converted = pd.to_numeric(original, errors="coerce")
        failure = original.notna() & converted.isna()
        clean[name][f"flag_{column.lower()}_numeric_failure"] = failure
        clean[name][column] = converted
        numeric_checks.append({
            "dataset": name, "column": column,
            "conversion_failures": int(failure.sum()),
        })

display(pd.DataFrame(numeric_checks))


## 7. Add transparent quality flags

Questionable values are retained. Coordinate checks distinguish missing, zero, globally impossible, and outside a broad Los Angeles bounding box. The LA-box flag is a review aid, not proof that a coordinate is wrong.


In [ ]:
def add_quality_flags(frame, dataset_name):
    if {"LAT", "LON"}.issubset(frame.columns):
        lat, lon = frame["LAT"], frame["LON"]
        frame["flag_missing_coordinates"] = lat.isna() | lon.isna()
        frame["flag_zero_coordinates"] = lat.eq(0) | lon.eq(0)
        frame["flag_globally_impossible_coordinates"] = (
            (~lat.between(-90, 90) | ~lon.between(-180, 180))
            & lat.notna() & lon.notna()
        )
        frame["flag_coordinates_outside_la_box"] = (
            lat.notna() & lon.notna()
            & ~frame["flag_zero_coordinates"]
            & ~frame["flag_globally_impossible_coordinates"]
            & (~lat.between(33.5, 34.9) | ~lon.between(-119.0, -117.5))
        )
        frame["valid_coordinates"] = ~(
            frame["flag_missing_coordinates"]
            | frame["flag_zero_coordinates"]
            | frame["flag_globally_impossible_coordinates"]
            | frame["flag_coordinates_outside_la_box"]
        )

    if "Area_No" in frame:
        frame["flag_missing_area"] = frame["Area_No"].isna()
        frame["flag_invalid_area"] = frame["Area_No"].notna() & ~frame["Area_No"].between(1, 21)
        frame["flag_noninteger_area"] = frame["Area_No"].notna() & frame["Area_No"].mod(1).ne(0)

    if "Report_District_No" in frame:
        district = frame["Report_District_No"]
        frame["flag_missing_district"] = district.isna()
        frame["flag_invalid_district"] = district.notna() & ~district.between(100, 2199)
        frame["flag_noninteger_district"] = district.notna() & district.mod(1).ne(0)
        frame["district_area"] = (district // 100).astype("Int64")
        frame["flag_area_district_mismatch"] = (
            frame.get("Area_No", pd.Series(index=frame.index, dtype="float64")).notna()
            & district.notna()
            & frame["Area_No"].ne(frame["district_area"])
        )

    if "Date_Occurred" in frame:
        frame["flag_occurrence_before_2000"] = frame["Date_Occurred"].notna() & frame["Date_Occurred"].lt("2000-01-01")
    if {"Date_Reported", "Date_Occurred"}.issubset(frame.columns):
        frame["flag_report_before_occurrence"] = (
            frame["Date_Reported"].notna() & frame["Date_Occurred"].notna()
            & frame["Date_Reported"].lt(frame["Date_Occurred"])
        )

    if dataset_name == "nibrs":
        for column, minimum in {"TotalOffenseCount": 1, "TotalVictimCount": 0}.items():
            if column in frame:
                frame[f"flag_invalid_{column.lower()}"] = frame[column].notna() & (
                    frame[column].lt(minimum) | frame[column].mod(1).ne(0)
                )
    if dataset_name == "legacy" and "Victim_Age" in frame:
        frame["flag_suspicious_victim_age"] = frame["Victim_Age"].notna() & ~frame["Victim_Age"].between(0, 120)
    return frame

for name in clean:
    clean[name] = add_quality_flags(clean[name], name)


## 8. Produce a quality summary and reconcile row counts

Assertions stop the notebook if source rows disappear, the source snapshots change, or output keys unexpectedly become less complete.


In [ ]:
quality_rows = []
for name, frame in clean.items():
    for column in [c for c in frame if c.startswith("flag_")]:
        quality_rows.append({
            "dataset": name,
            "quality_check": column,
            "flagged_rows": int(frame[column].fillna(False).sum()),
            "percent_of_rows": round(frame[column].fillna(False).mean() * 100, 3),
        })
quality_summary = pd.DataFrame(quality_rows).sort_values(["dataset", "flagged_rows"], ascending=[True, False])
display(quality_summary)

reconciliation = []
for name in raw:
    assert len(clean[name]) == len(raw[name]), f"{name}: row count changed"
    assert raw[name].shape[0] == len(raw[name]), f"{name}: source snapshot changed"
    reconciliation.append({
        "dataset": name,
        "rows_before": len(raw[name]),
        "rows_after": len(clean[name]),
        "rows_added_or_removed": len(clean[name]) - len(raw[name]),
        "columns_before": raw[name].shape[1],
        "columns_after": clean[name].shape[1],
    })
reconciliation = pd.DataFrame(reconciliation)
display(reconciliation)


## 9. Verify difficult records with original-to-cleaned comparisons

The sample deliberately prioritizes flagged or incomplete rows. It is an inspection sample, not a statistically representative sample.


In [ ]:
def difficult_sample(frame, key, size=25, seed=42):
    flag_columns = [column for column in frame if column.startswith("flag_")]
    difficulty = frame[flag_columns].fillna(False).sum(axis=1) if flag_columns else pd.Series(0, index=frame.index)
    difficult = difficulty[difficulty.gt(0)].sort_values(ascending=False).index.tolist()
    selected = difficult[:size]
    if len(selected) < size:
        remainder = frame.index.difference(selected)
        selected += frame.loc[remainder].sample(min(size - len(selected), len(remainder)), random_state=seed).index.tolist()
    sample = frame.loc[selected].copy()
    sample.insert(0, "sample_reason", [
        ", ".join(column for column in flag_columns if bool(frame.at[index, column])) or "deterministic fill"
        for index in selected
    ])
    preferred = ["sample_reason", key, "CaseNo", "Date_Reported", "Date_Occurred", "Time_Occurred",
                 "Area_No", "Area_Name", "Report_District_No", "LAT", "LON", "Victim_Age",
                 "TotalOffenseCount", "TotalVictimCount"]
    return sample[[column for column in preferred if column in sample] +
                  [column for column in flag_columns if column in sample]]

samples = {
    name: difficult_sample(frame, KEYS[name])
    for name, frame in clean.items()
}
for name, sample in samples.items():
    print(f"\n{name.upper()} difficult-case sample")
    display(sample.head(10))


## 10. Export cleaned data, audit evidence, and review samples

CSV exports use ISO-style datetime text. The JSON decision log records exactly what was changed and what was intentionally preserved.


In [ ]:
DECISION_LOG = [
    {"rule": "Preserve source", "decision": "Clean independent copies; do not modify raw snapshots."},
    {"rule": "Keep datasets separate", "decision": "Do not merge because row grain and coding systems differ."},
    {"rule": "Text", "decision": "Trim outer whitespace, collapse repeated whitespace, and convert blanks to missing."},
    {"rule": "Dates and time", "decision": "Parse dates, validate HHMM, and derive occurrence hour; flag failures."},
    {"rule": "Missing data", "decision": "Preserve missing values; do not substitute zero or guess categories."},
    {"rule": "Duplicates", "decision": "Report exact duplicates and repeated IDs; do not drop automatically."},
    {"rule": "Questionable values", "decision": "Retain records and add transparent flags for review."},
    {"rule": "Coordinates", "decision": "Flag missing, zero, impossible, and outside-LA-box coordinates."},
]

OUTPUT_FILES = {
    "nibrs_clean": OUTPUT_DIR / "cleaned_LAPD_NIBRS_Offenses_Dataset.csv",
    "legacy_clean": OUTPUT_DIR / "cleaned_Crime_Data_from_2020_to_2024.csv",
    "quality_summary": OUTPUT_DIR / "data_quality_summary.csv",
    "reconciliation": OUTPUT_DIR / "row_reconciliation.csv",
    "nibrs_sample": OUTPUT_DIR / "nibrs_difficult_cases_sample.csv",
    "legacy_sample": OUTPUT_DIR / "legacy_difficult_cases_sample.csv",
    "decision_log": OUTPUT_DIR / "cleaning_decision_log.json",
}

clean["nibrs"].to_csv(OUTPUT_FILES["nibrs_clean"], index=False, date_format="%Y-%m-%d %H:%M:%S")
clean["legacy"].to_csv(OUTPUT_FILES["legacy_clean"], index=False, date_format="%Y-%m-%d %H:%M:%S")
quality_summary.to_csv(OUTPUT_FILES["quality_summary"], index=False)
reconciliation.to_csv(OUTPUT_FILES["reconciliation"], index=False)
samples["nibrs"].to_csv(OUTPUT_FILES["nibrs_sample"], index=False)
samples["legacy"].to_csv(OUTPUT_FILES["legacy_sample"], index=False)
OUTPUT_FILES["decision_log"].write_text(json.dumps(DECISION_LOG, indent=2), encoding="utf-8")

pd.DataFrame([
    {"output": name, "path": str(path), "exists": path.exists(), "size_mb": round(path.stat().st_size / 1_048_576, 2)}
    for name, path in OUTPUT_FILES.items()
])


## 11. Final validation

This cell verifies that outputs exist, cleaned CSV row counts match the source data, and key fields are preserved. It reads only the exported headers and row counts, not duplicate full copies into memory.


In [ ]:
for path in OUTPUT_FILES.values():
    assert path.exists(), f"Missing output: {path}"

def csv_data_row_count(path):
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for _ in handle) - 1

assert csv_data_row_count(OUTPUT_FILES["nibrs_clean"]) == len(raw["nibrs"])
assert csv_data_row_count(OUTPUT_FILES["legacy_clean"]) == len(raw["legacy"])

for name, key in KEYS.items():
    output_key = clean[name][key]
    assert int(output_key.isna().sum()) == int(raw[name][key].isna().sum())

print("All validation checks passed.")
print("Cleaned files and audit evidence were written to:", OUTPUT_DIR.resolve())
display(reconciliation)


## Remaining limitations

- These files describe recorded offenses, not all harm or all incidents in Los Angeles.
- The NIBRS and legacy files have different row definitions and should not be stacked or compared as if they were identical.
- Flags identify values needing review; they do not prove that a source record is incorrect.
- The broad Los Angeles coordinate box is an approximate screening rule, not a boundary validation.
- Missing information is preserved because it cannot be safely inferred from the available columns.
- Descriptive counts are not population-adjusted risk measures and do not establish causes or resource effectiveness.
